In [1]:
import pandas as pd
import numpy as np
import random

In [8]:
ef = pd.read_csv('emission_factors.csv', on_bad_lines='skip')
print(ef.head(21))

                  Activity                 Sub_Type  \
0          Video Streaming                       4K   
1          Video Streaming                    1080p   
2          Video Streaming                     720p   
3          Video Streaming                SD/Mobile   
4         Short-form Video  Autoplay (Reels/Shorts)   
5   Social Media Scrolling   Text-heavy (Twitter/X)   
6              Video Calls               Camera Off   
7                    Email               Plain Text   
8                    Email   With Attachment (>1MB)   
9            Cloud Storage                   Upload   
10           Cloud Storage                 Download   
11         Music Streaming   Normal (Spotify/Gaana)   
12           AI Tool Usage    LLM Query (GPT-scale)   
13           Online Gaming           Mobile (Local)   

    Emission_Factor_Global_kgCO2  India_Adjusted_Factor_kgCO2         Unit  \
0                        0.10000                      0.16700     kgCO2/hr   
1                 

In [16]:
ef_netflix_4k    = float(ef[ef['Sub_Type'] == '4K']['India_Adjusted_Factor_kgCO2'].values[0])
ef_netflix_1080p = float(ef[ef['Sub_Type'] == '1080p']['India_Adjusted_Factor_kgCO2'].values[0])
ef_netflix_720p  = float(ef[ef['Sub_Type'] == '720p']['India_Adjusted_Factor_kgCO2'].values[0])
ef_netflix_sd    = float(ef[ef['Sub_Type'] == 'SD/Mobile']['India_Adjusted_Factor_kgCO2'].values[0])
ef_instagram     = float(ef[ef['Sub_Type'] == 'Autoplay (Reels/Shorts)']['India_Adjusted_Factor_kgCO2'].values[0])
ef_call_on       = ef_netflix_720p # Proxy for video calls with camera on
ef_call_off      = float(ef[ef['Sub_Type'] == 'Camera Off']['India_Adjusted_Factor_kgCO2'].values[0])
ef_email_plain   = float(ef[ef['Sub_Type'] == 'Plain Text']['India_Adjusted_Factor_kgCO2'].values[0])
ef_email_attach  = float(ef[ef['Sub_Type'] == 'With Attachment (>1MB)']['India_Adjusted_Factor_kgCO2'].values[0])
ef_cloud_upload  = float(ef[ef['Sub_Type'] == 'Upload']['India_Adjusted_Factor_kgCO2'].values[0])
ef_music         = float(ef[ef['Sub_Type'] == 'Normal (Spotify/Gaana)']['India_Adjusted_Factor_kgCO2'].values[0])
ef_ai_query      = float(ef[ef['Sub_Type'] == 'LLM Query (GPT-scale)']['India_Adjusted_Factor_kgCO2'].values[0])
ef_gaming_cloud  = ef_netflix_1080p # Proxy for cloud gaming
ef_gaming_mobile = float(ef[ef['Sub_Type'] == 'Mobile (Local)']['India_Adjusted_Factor_kgCO2'].values[0])

print("Emission factors loaded successfully")
print(f"Netflix 4K: {ef_netflix_4k} kgCO2/hr")
print(f"Instagram (Reels/Shorts): {ef_instagram} kgCO2/hr")
print(f"Video call camera on (proxy): {ef_call_on} kgCO2/hr")
print(f"Video call camera off: {ef_call_off} kgCO2/hr")
print(f"AI query: {ef_ai_query} kgCO2/query")
print(f"Cloud Gaming (proxy): {ef_gaming_cloud} kgCO2/hr")
print(f"Mobile Gaming: {ef_gaming_mobile} kgCO2/hr")

Emission factors loaded successfully
Netflix 4K: 0.167 kgCO2/hr
Instagram (Reels/Shorts): 0.1 kgCO2/hr
Video call camera on (proxy): 0.06 kgCO2/hr
Video call camera off: 0.01 kgCO2/hr
AI query: 0.00722 kgCO2/query
Cloud Gaming (proxy): 0.092 kgCO2/hr
Mobile Gaming: 0.013 kgCO2/hr


In [10]:
print(ef['Sub_Type'].unique())

['4K' '1080p' '720p' 'SD/Mobile' 'Autoplay (Reels/Shorts)'
 'Text-heavy (Twitter/X)' 'Camera Off' 'Plain Text'
 'With Attachment (>1MB)' 'Upload' 'Download' 'Normal (Spotify/Gaana)'
 'LLM Query (GPT-scale)' 'Mobile (Local)']


In [12]:
age_groups   = ['13-18', '19-25', '26-35', '36-50', '51+']
city_tiers   = ['Metro', 'Tier2', 'Rural']
professions  = ['Student', 'IT Professional', 'Business', 'Teacher', 'Homemaker', 'Gig Worker']
devices      = ['Mobile only', 'Laptop only', 'Both mobile and laptop']
income_levels = ['Low', 'Middle', 'High']

# Probability weights — based on TRAI 2023 Indian internet user statistics
# More young users, more metro users, more mobile users
age_weights      = [0.15, 0.30, 0.25, 0.20, 0.10]
city_weights     = [0.45, 0.35, 0.20]
profession_weights = [0.25, 0.20, 0.15, 0.15, 0.15, 0.10]
device_weights   = [0.55, 0.25, 0.20]
income_weights   = [0.40, 0.45, 0.15]

In [13]:
def get_behavior_profile(age_group, profession, device, income):

    # Base hours available for digital activity per day
    # Students and IT workers use internet most

    if profession == 'Student':
        netflix_hrs    = round(random.uniform(1.5, 4.5), 1)
        instagram_hrs  = round(random.uniform(1.5, 3.5), 1)
        call_on_hrs    = round(random.uniform(0.5, 2.0), 1)
        call_off_hrs   = round(random.uniform(0.0, 1.0), 1)
        emails_plain   = random.randint(2, 15)
        emails_attach  = random.randint(0, 5)
        cloud_gb       = round(random.uniform(0.1, 1.5), 2)
        music_hrs      = round(random.uniform(1.0, 3.0), 1)
        ai_queries     = random.randint(5, 30)
        gaming_cloud   = round(random.uniform(0.0, 1.5), 1)
        gaming_mobile  = round(random.uniform(0.5, 2.5), 1)

    elif profession == 'IT Professional':
        netflix_hrs    = round(random.uniform(0.5, 2.5), 1)
        instagram_hrs  = round(random.uniform(0.5, 1.5), 1)
        call_on_hrs    = round(random.uniform(2.0, 5.0), 1)
        call_off_hrs   = round(random.uniform(0.5, 2.0), 1)
        emails_plain   = random.randint(20, 60)
        emails_attach  = random.randint(5, 20)
        cloud_gb       = round(random.uniform(1.0, 5.0), 2)
        music_hrs      = round(random.uniform(1.0, 4.0), 1)
        ai_queries     = random.randint(15, 60)
        gaming_cloud   = round(random.uniform(0.0, 1.0), 1)
        gaming_mobile  = round(random.uniform(0.0, 1.0), 1)

    elif profession == 'Homemaker':
        netflix_hrs    = round(random.uniform(2.0, 5.0), 1)
        instagram_hrs  = round(random.uniform(1.0, 3.0), 1)
        call_on_hrs    = round(random.uniform(0.5, 1.5), 1)
        call_off_hrs   = round(random.uniform(0.0, 0.5), 1)
        emails_plain   = random.randint(1, 10)
        emails_attach  = random.randint(0, 3)
        cloud_gb       = round(random.uniform(0.0, 0.5), 2)
        music_hrs      = round(random.uniform(1.5, 4.0), 1)
        ai_queries     = random.randint(0, 10)
        gaming_cloud   = round(random.uniform(0.0, 0.0), 1)
        gaming_mobile  = round(random.uniform(0.5, 2.0), 1)

    elif profession == 'Business':
        netflix_hrs    = round(random.uniform(0.5, 2.0), 1)
        instagram_hrs  = round(random.uniform(0.5, 2.0), 1)
        call_on_hrs    = round(random.uniform(1.0, 4.0), 1)
        call_off_hrs   = round(random.uniform(0.5, 2.0), 1)
        emails_plain   = random.randint(15, 50)
        emails_attach  = random.randint(5, 25)
        cloud_gb       = round(random.uniform(0.5, 3.0), 2)
        music_hrs      = round(random.uniform(0.5, 2.0), 1)
        ai_queries     = random.randint(5, 25)
        gaming_cloud   = round(random.uniform(0.0, 0.5), 1)
        gaming_mobile  = round(random.uniform(0.0, 0.5), 1)

    else:  # Teacher, Gig Worker — moderate usage
        netflix_hrs    = round(random.uniform(1.0, 3.0), 1)
        instagram_hrs  = round(random.uniform(0.5, 2.0), 1)
        call_on_hrs    = round(random.uniform(0.5, 2.0), 1)
        call_off_hrs   = round(random.uniform(0.0, 1.0), 1)
        emails_plain   = random.randint(5, 25)
        emails_attach  = random.randint(1, 10)
        cloud_gb       = round(random.uniform(0.1, 1.5), 2)
        music_hrs      = round(random.uniform(1.0, 3.0), 1)
        ai_queries     = random.randint(2, 20)
        gaming_cloud   = round(random.uniform(0.0, 0.5), 1)
        gaming_mobile  = round(random.uniform(0.0, 1.5), 1)

    # Income modifier — wealthier users stream in higher quality
    if income == 'Low':
        netflix_quality = 'SD'
        netflix_hrs = netflix_hrs * 0.6  # less usage overall
    elif income == 'Middle':
        netflix_quality = '1080p'
    else:
        netflix_quality = '4K'

    return {
        'netflix_hrs': netflix_hrs,
        'netflix_quality': netflix_quality,
        'instagram_hrs': instagram_hrs,
        'call_on_hrs': call_on_hrs,
        'call_off_hrs': call_off_hrs,
        'emails_plain': emails_plain,
        'emails_attach': emails_attach,
        'cloud_upload_gb': cloud_gb,
        'music_hrs': music_hrs,
        'ai_queries': ai_queries,
        'gaming_cloud_hrs': gaming_cloud,
        'gaming_mobile_hrs': gaming_mobile
    }

In [14]:
def calculate_weekly_carbon(behavior):

    # Select netflix emission factor based on quality
    if behavior['netflix_quality'] == '4K':
        ef_netflix = ef_netflix_4k
    elif behavior['netflix_quality'] == '1080p':
        ef_netflix = ef_netflix_1080p
    else:
        ef_netflix = ef_netflix_sd

    # Daily carbon in kgCO2
    daily_carbon = (
        behavior['netflix_hrs']        * ef_netflix +
        behavior['instagram_hrs']      * ef_instagram +
        behavior['call_on_hrs']        * ef_call_on +
        behavior['call_off_hrs']       * ef_call_off +
        behavior['emails_plain']       * ef_email_plain +
        behavior['emails_attach']      * ef_email_attach +
        behavior['cloud_upload_gb']    * ef_cloud_upload +
        behavior['music_hrs']          * ef_music +
        behavior['ai_queries']         * ef_ai_query +
        behavior['gaming_cloud_hrs']   * ef_gaming_cloud +
        behavior['gaming_mobile_hrs']  * ef_gaming_mobile
    )

    weekly_carbon = daily_carbon * 7
    return round(weekly_carbon, 4)

# BDCI normalization — converts raw kgCO2/week to 0-100 score
# Based on realistic Indian user range: min ~0.3 kg/week, max ~15 kg/week
BDCI_MIN = 0.3
BDCI_MAX = 15.0

def compute_bdci(weekly_carbon):
    score = (weekly_carbon - BDCI_MIN) / (BDCI_MAX - BDCI_MIN) * 100
    return round(min(max(score, 0), 100), 2)  # clamp between 0 and 100

In [17]:
np.random.seed(42)  # for reproducibility — same results every run
random.seed(42)

records = []

for i in range(5000):

    # Sample user profile
    age      = random.choices(age_groups, weights=age_weights)[0]
    city     = random.choices(city_tiers, weights=city_weights)[0]
    prof     = random.choices(professions, weights=profession_weights)[0]
    device   = random.choices(devices, weights=device_weights)[0]
    income   = random.choices(income_levels, weights=income_weights)[0]

    # Get behavior
    behavior = get_behavior_profile(age, prof, device, income);

    # Compute carbon and BDCI
    weekly_carbon = calculate_weekly_carbon(behavior);
    bdci_score    = compute_bdci(weekly_carbon);

    # Classify into categories (for classification model in Step 4)
    if bdci_score < 25:
        bdci_category = 'Low'
    elif bdci_score < 50:
        bdci_category = 'Moderate'
    elif bdci_score < 75:
        bdci_category = 'High'
    else:
        bdci_category = 'Critical'

    records.append({
        'user_id':            f'U{i+1:04d}',
        'age_group':          age,
        'city_tier':          city,
        'profession':         prof,
        'device_type':        device,
        'income_level':       income,
        'netflix_hrs_day':    behavior['netflix_hrs'],
        'netflix_quality':    behavior['netflix_quality'],
        'instagram_hrs_day':  behavior['instagram_hrs'],
        'call_on_hrs_day':    behavior['call_on_hrs'],
        'call_off_hrs_day':   behavior['call_off_hrs'],
        'emails_plain_day':   behavior['emails_plain'],
        'emails_attach_day':  behavior['emails_attach'],
        'cloud_upload_gb_day':behavior['cloud_upload_gb'],
        'music_hrs_day':      behavior['music_hrs'],
        'ai_queries_day':     behavior['ai_queries'],
        'gaming_cloud_hrs':   behavior['gaming_cloud_hrs'],
        'gaming_mobile_hrs':  behavior['gaming_mobile_hrs'],
        'weekly_carbon_kg':   weekly_carbon,
        'bdci_score':         bdci_score,
        'bdci_category':      bdci_category
    })

df = pd.DataFrame(records)
print(f"Dataset shape: {df.shape}")
print(df.head())
print(f"\nBDCI Score distribution:")
print(df['bdci_category'].value_counts())

Dataset shape: (5000, 21)
  user_id age_group city_tier       profession             device_type  \
0   U0001     26-35     Metro  IT Professional             Mobile only   
1   U0002     36-50     Metro  IT Professional             Mobile only   
2   U0003     36-50     Tier2         Business  Both mobile and laptop   
3   U0004     36-50     Metro         Business  Both mobile and laptop   
4   U0005       51+     Tier2         Business             Laptop only   

  income_level  netflix_hrs_day netflix_quality  instagram_hrs_day  \
0       Middle             1.90           1080p                1.4   
1         High             2.00              4K                0.7   
2          Low             0.78              SD                1.7   
3          Low             0.60              SD                1.5   
4       Middle             1.70           1080p                0.8   

   call_on_hrs_day  ...  emails_plain_day  emails_attach_day  \
0              2.3  ...                21   

In [18]:
print("=== Dataset Validation ===")
print(f"Total users: {len(df)}")
print(f"BDCI Score range: {df['bdci_score'].min()} to {df['bdci_score'].max()}")
print(f"Average weekly carbon: {df['weekly_carbon_kg'].mean():.3f} kgCO2")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"\nCity tier distribution:")
print(df['city_tier'].value_counts())
print(f"\nProfession distribution:")
print(df['profession'].value_counts())

# Save to CSV
df.to_csv('behavioral_carbon_dataset.csv', index=False)
print("\nDataset saved as behavioral_carbon_dataset.csv")

=== Dataset Validation ===
Total users: 5000
BDCI Score range: 18.37 to 100.0
Average weekly carbon: 10.994 kgCO2
Missing values: 0

City tier distribution:
city_tier
Metro    2327
Tier2    1729
Rural     944
Name: count, dtype: int64

Profession distribution:
profession
Student            1254
IT Professional    1038
Teacher             793
Business            735
Homemaker           735
Gig Worker          445
Name: count, dtype: int64

Dataset saved as behavioral_carbon_dataset.csv
